In [3]:
import matplotlib.pyplot as plt
import tensorflow as tf
from keras.src.legacy.preprocessing.image import ImageDataGenerator
from dotenv import load_dotenv
from PIL import Image
import pandas as pd
import os

Image.MAX_IMAGE_PIXELS = None
load_dotenv()

# Parameters
img_size = 224
lr = 1e-4
num_classes = 2
num_epochs = 5

# Environment variables
TRAIN_DIR = os.getenv('TRAIN_DIR')
TEST_DIR = os.getenv('TEST_DIR')
CSV_PATH = os.getenv('CSV_PATH')
MODEL_PATH = os.getenv('MODEL_PATH')
OUTPUT_PATH = "/output/plotting/"


def check_images(directory):
    for root, _, files in os.walk(directory):
        for file in files:
            try:
                img_paths = os.path.join(root, file)
                img = Image.open(img_paths)
                img.verify()
            except (IOError, SyntaxError) as e:
                print(f"Corrupted or unreadable image: {img_paths}, {e}")


check_images(TRAIN_DIR)
check_images(TEST_DIR)

# Data augmentation for training
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(img_size, img_size),
    batch_size=64,
    class_mode='categorical',
    subset='training',
    color_mode='grayscale',
    seed=123,
    shuffle=True,
)

validation_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(img_size, img_size),
    batch_size=64,
    class_mode='categorical',
    subset='validation',
    color_mode='grayscale',
    seed=123,
    shuffle=True,
)

test_datagen = ImageDataGenerator(rescale=1.0 / 255.0)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(img_size, img_size),
    batch_size=64,
    class_mode=None,
    color_mode='grayscale',
    shuffle=False
)

# Define the CNN model 
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(img_size, img_size, 1)),

    # First block
    tf.keras.layers.Conv2D(32, (3, 3), padding='same', activation="relu"),
    tf.keras.layers.MaxPooling2D((2, 2), strides=2),

    # Second block
    tf.keras.layers.Conv2D(64, (3, 3), padding='same', activation="relu"),
    tf.keras.layers.MaxPooling2D((2, 2), strides=2),

    # Third block
    tf.keras.layers.Conv2D(128, (3, 3), padding='same', activation="relu"),
    tf.keras.layers.MaxPooling2D((2, 2), strides=2),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(256, activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(num_classes, activation="softmax")
])

lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=lr,
    decay_steps=10000,
    decay_rate=0.9
)

# Optimizer
optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule)

# Compile the model
model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
history = model.fit(
    train_generator,
    epochs=num_epochs,
    validation_data=validation_generator,
)

# Save the trained model
model.save(MODEL_PATH)

# Save training history to CSV
pd.DataFrame(history.history).to_csv(CSV_PATH, index=False)


# Plot training and validation accuracy and loss
def plot_history(history_data, save_path):
    plt.figure(figsize=(12, 4))

    # Accuracy subplot
    plt.subplot(1, 2, 1)
    plt.plot(history_data.history['accuracy'])
    plt.plot(history_data.history['val_accuracy'])
    plt.title('Model accuracy')
    plt.ylabel('Accuracy')
    plt.xlabel('Epoch')
    plt.legend(['Training', 'Validation'], loc='upper left')

    # Loss subplot
    plt.subplot(1, 2, 2)
    plt.plot(history_data.history['loss'])
    plt.plot(history_data.history['val_loss'])
    plt.title('Model loss')
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend(['Training', 'Validation'], loc='upper left')

    # Save the figure
    plt.savefig(save_path)
    plt.show()


plot_history(history, os.path.join(OUTPUT_PATH, 'training_history.jpg'))

Train Directory: /Users/sochea/Documents/MITE/dataset/training
Test Directory: /Users/sochea/Documents/MITE/dataset/testing
CSV Path: /Users/sochea/Documents/MITE/dataset/save_model/cnn_training_history.csv
Model Path: /Users/sochea/Documents/MITE/dataset/save_model/new_trained_model.h5
